# eBOSS post-starburst selection

In this first notebook, we walk through the selection methods that allow us to take a parent sample of ~2 million galaxies from SDSS-IV/[eBOSS](https://www.sdss4.org/surveys/eboss/) and trim it down to a subsample of young and classic post-starburst candidates

In [1]:
import numpy as np               # to do math
from astropy.io import fits      # handline .fits files
import pandas as pd              # table format
import os                        # universal os


## Loading data

In [2]:
data_folder_name = 'data'  # folder containing the data file
filename_summary = 'eboss_dr17_summary_dr2_v1.fit'  # main data file
filename_weights = 'eboss_dr17_weights_dr2_v1.fit'  # weights file

# accessing file from current directory
file_path_summary = os.path.join('..', data_folder_name, filename_summary)
file_path_weights = os.path.join('..', data_folder_name, filename_weights)

file_path_summary
file_path_weights

'../data/eboss_dr17_weights_dr2_v1.fit'

To access the data, we will use `astropy` `fits` functions. Once opened, our data is stored in a "FITS table". The table header has info on the dimensions (# of rows and columns) and the column names. 

In [3]:
# hdu: "header data unit"
# contains information on the format and contents of our fits file
hdu_summary = fits.open(file_path_summary)
hdu_weights = fits.open(file_path_weights)

# displaying the header of our fits file
hdu_summary[1].header 

print(hdu_summary[1].columns.names) # checking column names because I am unsure

['PLATE', 'MJD', 'FIBER', 'PMF_STRING', 'Z', 'Z_ERR', 'RA', 'DEC', 'EBV_MW', 'VDISP', 'VDISP_ERR', 'TARGET_TEXT', 'TARGET_CLASS', 'MGII_2800_FLUX', 'MGII_2800_FLUX_ERR', 'MGII_2800_EW', 'NEV_3347_FLUX', 'NEV_3347_FLUX_ERR', 'NEV_3347_EW', 'NEV_3426_FLUX', 'NEV_3426_FLUX_ERR', 'NEV_3426_EW', 'OII_3727_FLUX', 'OII_3727_FLUX_ERR', 'OII_3727_EW', 'NEIII_3869_FLUX', 'NEIII_3869_FLUX_ERR', 'NEIII_3869_EW', 'H_DELTA_FLUX', 'H_DELTA_FLUX_ERR', 'H_DELTA_EW', 'H_GAMMA_FLUX', 'H_GAMMA_FLUX_ERR', 'H_GAMMA_EW', 'H_BETA_FLUX', 'H_BETA_FLUX_ERR', 'H_BETA_EW', 'H_BETA_SIGMA', 'H_BETA_SIGMA_ERR', 'H_BETA_SIGMA_CORR', 'OIII_4363_FLUX', 'OIII_4363_FLUX_ERR', 'OIII_4363_EW', 'HEII_4686_FLUX', 'HEII_4686_FLUX_ERR', 'HEII_4686_EW', 'OIII_4959_FLUX', 'OIII_4959_FLUX_ERR', 'OIII_4959_EW', 'OIII_5007_FLUX', 'OIII_5007_FLUX_ERR', 'OIII_5007_EW', 'OIII_5007_SIGMA', 'OIII_5007_SIGMA_ERR', 'OIII_5007_SIGMA_CORR', 'OI_6300_FLUX', 'OI_6300_FLUX_ERR', 'OI_6300_EW', 'FEX_6374_FLUX', 'FEX_6374_FLUX_ERR', 'FEX_6374_EW',

As you can see, this table has a LOT of columns... to make things simple, we will extract the columns that we need and ignore the rest.

The main column values we are interested in are 

index measurements: 

- `DN4000` --> 4000 Angstrom break 
- `LICK_HDA` --> H-delta absorption
- `BALMERBREAK` --> Balmer break

their associated errors:

- `DN4000_ERR`
- `LICK_HDA_ERR`
- `BALMERBREAK_ERR`

line measurements:

- `H_ALPHA_EW`  --> strength of H-alpha emission
- `H_BETA_EW` --> strength of H-beta emission

and ionization source classification:

- `BPT_CLASS` --> classification on the BPT diagram

We will also save basic stuff about these eBOSS sources, such as redshift and identification information.

In [4]:
# ... this cell might take a minute to run ...

# initializing a DataFrame
eBOSS_summary = pd.DataFrame()

# ------- setting the columns and rows for our table --------

# general info on objects
eBOSS_summary['PLATE'] = list(hdu_summary[1].data['PLATE'])                      
eBOSS_summary['MJD'] = list(hdu_summary[1].data['MJD'])  
eBOSS_summary['FIBER'] = list(hdu_summary[1].data['FIBER'])     
# PLATE, MJD, and FIBER numbers combined into an identification string
eBOSS_summary['PMF_STRING'] = list(hdu_summary[1].data['PMF_STRING'])    
# redshift of galaxy
eBOSS_summary['Z'] = list(hdu_summary[1].data['Z'])  
eBOSS_summary['Z_ERR'] = list(hdu_summary[1].data['Z_ERR'])  
# coordinates of galaxy 
eBOSS_summary['RA'] = list(hdu_summary[1].data['RA'])                                         
eBOSS_summary['DEC'] = list(hdu_summary[1].data['DEC']) 

# relevant measurements
eBOSS_summary['H_ALPHA_EW'] = list(hdu_summary[1].data['H_ALPHA_EW']) 
eBOSS_summary['H_BETA_EW'] = list(hdu_summary[1].data['H_BETA_EW']) 
eBOSS_summary['OII_EW'] = list(hdu_summary[1].data['OII_3727_EW'])
eBOSS_summary['D_4000'] = list(hdu_summary[1].data['DN4000']) 
eBOSS_summary['D_4000_ERR'] = list(hdu_summary[1].data['DN4000_ERR'])
eBOSS_summary['LICK_HDA'] = list(hdu_summary[1].data['LICK_HDA'])
eBOSS_summary['LICK_HDA_ERR'] = list(hdu_summary[1].data['LICK_HDA_ERR'])
eBOSS_summary['BALMERBREAK'] = list(hdu_summary[1].data['BALMERBREAK'])
eBOSS_summary['BALMERBREAK_ERR'] = list(hdu_summary[1].data['BALMERBREAK_ERR'])
eBOSS_summary['BPT_CLASS'] = list(hdu_summary[1].data['BPT_CLASS'])
eBOSS_summary['SN_MEDIAN_ALL'] = list(hdu_summary[1].data['SN_MEDIAN'])  # S/N continuum column

# --------------------------------------------------------------

# okay now we need to decode the byte strings if needed
if isinstance(eBOSS_summary['PMF_STRING'].iloc[0], bytes):
    eBOSS_summary['PMF_STRING'] = eBOSS_summary['PMF_STRING'].str.decode('utf-8')

# load in the template weights by converting multi-dimensional columns safely
data_weights = hdu_weights[1].data
dict_weights = {}

for name in data_weights.dtype.names:
    dict_weights[name] = list(data_weights[name])

df_weights = pd.DataFrame(dict_weights)

if isinstance(df_weights['PMF_STRING'].iloc[0], bytes):
    df_weights['PMF_STRING'] = df_weights['PMF_STRING'].str.decode('utf-8')

# merge measurements and SSP weights together into one catalog
eBOSS_catalog = pd.merge(eBOSS_summary, df_weights, on='PMF_STRING', suffixes=('', '_weights'))
eBOSS_catalog

,PLATE,MJD,FIBER,PMF_STRING,Z,Z_ERR,RA,DEC,H_ALPHA_EW,H_BETA_EW,...,LICK_HDA_ERR,BALMERBREAK,BALMERBREAK_ERR,BPT_CLASS,SN_MEDIAN_ALL,PLATE_weights,MJD_weights,FIBER_weights,Z_weights,FIT_WTS
0,3586,55181,2,03586-55181-0002,0.719691,0.000069,9.330078,-0.624116,-999.000000,6.830700,...,1.299937,1.611019,0.080973,U,2.404119,3586,55181,2,0.719691,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,3586,55181,6,03586-55181-0006,0.533117,0.000141,9.409157,-0.250058,2.060369,0.239696,...,1.792657,2.815180,0.180261,U,2.808001,3586,55181,6,0.533117,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,3586,55181,7,03586-55181-0007,0.474463,0.000134,9.360470,-0.212842,0.000000,0.510076,...,2.099126,2.208125,0.129288,U,2.430126,3586,55181,7,0.474463,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
3,3586,55181,8,03586-55181-0008,0.489547,0.000159,9.351932,-0.159622,2.960478,0.000000,...,2.426148,2.312129,0.152515,U,2.103377,3586,55181,8,0.489547,"[[2.349242e-05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,3586,55181,9,03586-55181-0009,0.489842,0.000225,9.452027,-0.105271,1.556909,0.766571,...,3.165549,2.845022,0.305743,U,1.702376,3586,55181,9,0.489842,"[[8.005477e-06, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1893690,12547,58928,985,12547-58928-0985,0.799789,0.000183,140.996347,2.055424,-999.000000,0.699077,...,2.805292,2.392276,0.226185,U,0.879351,12547,58928,985,0.799789,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1893691,12547,58928,986,12547-58928-0986,0.391036,0.000069,140.956507,2.087150,0.595381,0.788362,...,1.052939,1.982498,0.072412,U,5.748506,12547,58928,986,0.391036,"[[0.0, 6.611271e-05, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1893692,12547,58928,988,12547-58928-0988,0.024241,0.000021,140.953350,2.107938,114.652946,26.197693,...,7.886981,1.487158,0.719565,S,0.845627,12547,58928,988,0.024241,"[[0.0, 0.0, 0.0, 0.06918005, 0.0066976612, 0.0..."
1893693,12547,58928,991,12547-58928-0991,0.535690,0.000092,140.905378,2.599161,34.857849,8.350409,...,3.158643,1.942474,0.177461,L,2.496635,12547,58928,991,0.535690,"[[0.0, 0.0011064163, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Now we have a table with the data that we care about!

In [5]:
# Defining our 31 interesting galaxies
interesting_galaxies = [(8832, 918), (4221, 271), (11376, 234), (8497, 723), (7711, 557), (10750, 178), 
(9166, 760), (11114, 28), (10747, 766), (7388, 794), (7639, 684), (8233, 695), (8530, 222), 
(8491, 740), (8360, 94), (8381, 531), (10912, 812), (10232, 822), (10925, 294), (7572, 656), (11565, 847), (7683, 349),
(11336, 284), (7306, 754), (7735, 154), (7279, 396), (10938, 81), (6262, 911), (5967, 311), (8745, 354), (6453, 775)]

## Signal-to-noise and outlier cuts

We will set a reasonable signal-to-noise requirement for our sample ($\, S/N \geq 10$) and cut galaxies with unphysical values for any of our measurements.

In [6]:
final_quality_check = (

    (eBOSS_catalog['SN_MEDIAN_ALL'] >= 3.0) & 
    
    (eBOSS_catalog['D_4000_ERR'] != 0) &
    (eBOSS_catalog['LICK_HDA_ERR'] != 0) &
    (eBOSS_catalog['BALMERBREAK_ERR'] > 0) &

    (eBOSS_catalog['D_4000'] / eBOSS_catalog['D_4000_ERR'] >= 10) & 
    (eBOSS_catalog['BALMERBREAK'] / eBOSS_catalog['BALMERBREAK_ERR'] >= 10) &

    (eBOSS_catalog['LICK_HDA_ERR'] < 1.5) &
    (
        ((eBOSS_catalog['Z'] <= 0.49) & (eBOSS_catalog['H_ALPHA_EW'] != -999)) | 
        ((eBOSS_catalog['Z'] > 0.49) & ((eBOSS_catalog['H_BETA_EW'] != -999) | (eBOSS_catalog['OII_EW'] != -999)))
    )
)

eBOSS_catalog = eBOSS_catalog[final_quality_check]

print(f"Total rows remaining in final sample: {len(eBOSS_catalog)}")
final_matches = eBOSS_catalog[eBOSS_catalog.set_index(['PLATE', 'FIBER']).index.isin(interesting_galaxies)]
print(f"Total interesting galaxies found in final sample after signal-to-noise and invalid data cuts: {len(final_matches)} out of 31 interesting galaxies.")

Total rows remaining in final sample: 446662
Total interesting galaxies found in final sample after signal-to-noise and invalid data cuts: 31 out of 31 interesting galaxies.


Let's also remove any outliers in our dataset

In [7]:

standard_outliar_mask = (
    (eBOSS_catalog['LICK_HDA'] > -5) & (eBOSS_catalog['LICK_HDA'] < 13) & 
    (eBOSS_catalog['D_4000'] > 0) & (eBOSS_catalog['D_4000'] < 2.5) & 
    (eBOSS_catalog['BALMERBREAK'] > 0) & (eBOSS_catalog['BALMERBREAK'] < 3.5) 
)

line_outliar_mask = (
    ((eBOSS_catalog['Z'] <= 0.49) & 
     (eBOSS_catalog['H_ALPHA_EW'] > 0) & 
     (eBOSS_catalog['H_BETA_EW'] > 0)
    )
    |
    (
        (eBOSS_catalog['Z'] > 0.49) & 
        (
          (eBOSS_catalog['H_BETA_EW'] > 0) | 
          (eBOSS_catalog['OII_EW'] > 0)
        )
     )
 )
final_outliar_mask = standard_outliar_mask & line_outliar_mask

eBOSS_catalog = eBOSS_catalog[final_outliar_mask]

print(f"Total rows remaining in orginal catalog: {len(eBOSS_catalog)}")
outliar_matches = eBOSS_catalog[eBOSS_catalog.set_index(['PLATE', 'FIBER']).index.isin(interesting_galaxies)]
print(f"Interesting galaxies surviving the outliar mask: {len(outliar_matches)} out of 31")

Total rows remaining in orginal catalog: 234845
Interesting galaxies surviving the outliar mask: 31 out of 31


## Selection process 

We will select post-starburst galaxies using relationships between spectral features. Below is a starting point. We experiment with different selection methods as well.

The selection criteria for classic post-starburst galaxies are taken from Chen+2019 (see the figure below - the region outlined in black represents this cut). And the selection criterial for young post-starbursts are taken from Prof. Christy Tremonti's experimentations.

<img src="../plots/Chen+2019_figure.png" width="600">

In [8]:
# taken from Chen+2019 (region outlined in black)
# Redfining the classification masks here, making them log-safe and high redshift friendly
halpha_log_safe = np.log10(np.where(eBOSS_catalog['H_ALPHA_EW'] > 0, eBOSS_catalog['H_ALPHA_EW'], np.nan))

classic_psb_mask = (
    (eBOSS_catalog['LICK_HDA'] > 3.0) &

     # for lower redshift galaxies we will just use standard h-alpha criteria like before
     (((eBOSS_catalog['Z'] <= .58) &
       (eBOSS_catalog['H_ALPHA_EW'] < 10.0) &
       (halpha_log_safe < (0.23 * eBOSS_catalog['LICK_HDA'] - 0.46))) |

    # for higher redshift galaxies we will use H-beta cuts, or OII limits
      ((eBOSS_catalog['Z'] > 0.58) &
       ((eBOSS_catalog['H_BETA_EW'] < 3.0) | (eBOSS_catalog['OII_EW'] < 10.0))))
)

HALPHA_YOUNG_PSB_CUT = 7.0 # can be changed as needed! This is just the value Chrirsty suggested

young_psb_mask = (
    # 1.) Balmer Break and continuums for young PSBs 
    (eBOSS_catalog['BALMERBREAK'] > 0.9) & (eBOSS_catalog['BALMERBREAK'] < 1.9) &
    (eBOSS_catalog['LICK_HDA'] > (eBOSS_catalog['BALMERBREAK'] * 5.2 - 4.7)) &
    (eBOSS_catalog['BALMERBREAK'] > (eBOSS_catalog['D_4000'] * 7.0 - 6.5)) &
    
    # 2.) H-beta upper limit (CHANGED as of 7/29) # THIS CUT OUT 200 GALAXIES... ? is that appropriate...
    (eBOSS_catalog['H_BETA_EW'] < 5.0) &

    # 3.) redshift aware h-alpha cuts
    (
        ( # low redshift: h-alpha is in frame, must be strictly < 3.0 A
            (eBOSS_catalog['Z'] <= 0.49) & 
            (eBOSS_catalog['H_ALPHA_EW'] > -900) & 
            (eBOSS_catalog['H_ALPHA_EW'] < HALPHA_YOUNG_PSB_CUT)
        )
        |
        # high redshift: h-alpha is invalid/shifted out
        (eBOSS_catalog['Z'] > 0.49)
    )
)

# now we will apply these classifications to our new, larger catalog
eBOSS_catalog['PSB_CLASS'] = 'non_PSB'
eBOSS_catalog.loc[classic_psb_mask, 'PSB_CLASS'] = 'classic PSB'
eBOSS_catalog.loc[young_psb_mask, 'PSB_CLASS'] = 'young PSB'

# save the classified sample
eBOSS_catalog.to_csv('../classified_sample/classified_eBOSS_sample.csv', index=False)
print("Catalog classified and saved without warnings.")

# Print summary breakdown (after h-alpha cut updates)
print("Sample Classification Breakdown:")
print(eBOSS_catalog['PSB_CLASS'].value_counts())

Catalog classified and saved without warnings.
Sample Classification Breakdown:
PSB_CLASS
non_PSB        221800
classic PSB     12766
young PSB         279
Name: count, dtype: int64


We have saved our sample measurements and classifications in a .csv file and will use it in the next notebook: `02_eBOSS_data_analysis.ipynb`.

To experiement with different classification criteria, you can make changes to the cell above and a new .csv file will replace the current one.

In [61]:
# This code block was used to decide what cuts we should make based on S/N median values of our current sample
# isolate young PSB candidates
young_psbs = eBOSS_catalog[eBOSS_catalog['PSB_CLASS'] == 'young PSB']

# look at the S/N summary statistics for young PSBs
print("--- Young PSB S/N Continuum Stats ---")
print(young_psbs['SN_MEDIAN_ALL'].describe())


--- Young PSB S/N Continuum Stats ---
count    439.000000
mean       6.903247
std        7.390594
min        3.004357
25%        3.575935
50%        4.453612
75%        6.563415
max       54.673904
Name: SN_MEDIAN_ALL, dtype: float64
